# Ceftriaxone — Site-Deconfounding Diagnostic (LDA)

**Can we remove the site/instrument signal by projecting out its linear discriminative directions — instead of hard-zeroing bins?**

Fits LDA to the 4 site labels, projects out the top-1/2/3 site directions, and measures:
- `site_oob` (Random-Forest site OOB — want it to drop from ~1.0)
- `res_balacc` / `res_auc` (resistance OOB — want them preserved)
- a linear (LDA) site CV accuracy reference: is the site signal linearly separable?

Reuses the exact 06-03d data pipeline (same split, downsampling, preprocessing, SEED).

In [ ]:
# ── CONFIG ──
RUN_NAME = "06-03d-SiteDeconfound-Diagnostic"
TARGET_RUN = "01-Run"


In [ ]:
!pip install "flwr[simulation]" maldideepkit maldiamrkit --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier

from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldiamrkit.evaluation import stratified_species_drug_split

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)


In [ ]:
if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
else:
    DRYAD = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet")

DRUG_NAME = "Ceftriaxone"

PROJECT_DIR = DRYAD / "Processed/Processing/Analysis/06b-Ceftriaxone-E-coli" / RUN_NAME
RUN_DIR = PROJECT_DIR / TARGET_RUN
OUT_DIR = RUN_DIR / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SITES_PATHS = {
    "A": DRYAD / "Processed/Proc_DRIAMS-A" / DRUG_NAME / "data.csv",
    "B": DRYAD / "Processed/Proc_DRIAMS-B" / DRUG_NAME / "data.csv",
    "C": DRYAD / "Processed/Proc_DRIAMS-C" / DRUG_NAME / "data.csv",
    "D": DRYAD / "Processed/Proc_DRIAMS-D" / DRUG_NAME / "data.csv",
}
SITE_ORDER = ["A", "B", "C", "D"]
print(f"Drug: {DRUG_NAME}  |  Run: {RUN_DIR}")


In [ ]:
# ── Load Ceftriaxone — ALL species ──
raw_data = {}
for site, path in SITES_PATHS.items():
    df = pd.read_csv(path)
    bin_cols = [c for c in df.columns if c.startswith("bin_")]
    X = df[bin_cols].to_numpy(dtype="float32")
    y = df["label"].to_numpy(dtype="int64")
    species = df["species"].values
    raw_data[site] = (X, y, species)
    n_sp = len(np.unique(species))
    print(f"  Site {site}: {len(y)} samples ({n_sp} species)")
print(f"Total: {sum(len(raw_data[s][1]) for s in SITE_ORDER)}")


In [ ]:
# ── Per-site species-stratified 90/10 split (adaptive test_size) ──
client_train = {}; client_test = {}
species_train = {}; species_test = {}

for site in SITE_ORDER:
    X, y, sp = raw_data[site]
    n = len(y); idx = np.arange(n).reshape(-1,1)
    n_strata = len(set(zip(sp.tolist(), y.tolist())))
    test_size = max(0.10, (n_strata + 1) / n)
    itr, iv, _, _ = stratified_species_drug_split(idx, y, species=sp, test_size=test_size, random_state=SEED)
    itr = itr.flatten().astype(int); iv = iv.flatten().astype(int)
    client_train[site] = (X[itr], y[itr]); client_test[site] = (X[iv], y[iv])
    species_train[site] = sp[itr]; species_test[site] = sp[iv]
    print(f"  Site {site}: train={len(itr)} test={len(iv)} species={len(np.unique(sp[itr]))} (test_size={test_size:.2f})")


In [ ]:
# ── Downsample training data (keep all R, cap S in bad-ratio species) ──
TRAIN_DS_S_PER_R = 10   # global: all training data (bad-ratio species only)

def downsample_keep_idx(y, sp, s_per_r, bad_ratio=0.10, min_n=400, rng=None):
    rng = rng if rng is not None else np.random.default_rng(SEED)
    idx = np.arange(len(y))
    keep = idx[y == 1].tolist()
    for spec in np.unique(sp):
        m = sp == spec
        n_r = int((m & (y == 1)).sum()); n_s = int((m & (y == 0)).sum())
        total = n_r + n_s
        s_pos = idx[m & (y == 0)]
        if n_r > 0 and total > min_n and (n_r / total) < bad_ratio and n_s > n_r * s_per_r:
            keep.extend(rng.choice(s_pos, size=int(round(n_r * s_per_r)), replace=False).tolist())
        else:
            keep.extend(s_pos.tolist())
    return np.sort(np.array(keep, dtype=int))

_rng_ds = np.random.default_rng(SEED)
for site in SITE_ORDER:
    X, y = client_train[site]; sp = species_train[site]
    keep = downsample_keep_idx(y, sp, TRAIN_DS_S_PER_R, rng=_rng_ds)
    client_train[site] = (X[keep], y[keep]); species_train[site] = sp[keep]
    print(f"  Site {site}: kept {len(keep)}/{len(y)} train")


In [ ]:
# ── Per-site preprocessing ──
client_train_pp = {}
for site in SITE_ORDER:
    X_tr, y_tr = client_train[site]
    state = fit_input_transform(X_tr, "log1p+standardize")
    client_train_pp[site] = (apply_input_transform(X_tr, state), y_tr)
print("Per-site preprocessing done.")


In [ ]:
# ── LDA site deconfounding + evaluation ──
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import cross_val_score

X_all  = np.concatenate([client_train_pp[s][0] for s in SITE_ORDER])
y_site = np.concatenate([np.full(len(client_train_pp[s][0]), i) for i, s in enumerate(SITE_ORDER)])
y_res  = np.concatenate([client_train_pp[s][1] for s in SITE_ORDER])
print(f"Pooled: {len(X_all)} samples, 4 sites")

# reference: how linearly separable is site?
lda_ref = cross_val_score(LinearDiscriminantAnalysis(), X_all, y_site, cv=5, n_jobs=-1)
print(f"Linear (LDA) site CV accuracy: {lda_ref.mean():.4f} +/- {lda_ref.std():.4f}")

# fit LDA to get orthonormal site-discriminative directions
lda = LinearDiscriminantAnalysis(n_components=3).fit(X_all, y_site)
V, _ = np.linalg.qr(lda.scalings_)            # (6000, 3) orthonormal

def evaluate(Xc):
    rf_site = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, oob_score=True, random_state=SEED)
    rf_site.fit(Xc, y_site)
    site_oob = rf_site.oob_score_
    rf_res = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced", n_jobs=-1, oob_score=True, random_state=SEED)
    rf_res.fit(Xc, y_res)
    proba = rf_res.oob_decision_function_[:, 1]
    pred = (proba >= 0.5).astype(int)
    return site_oob, balanced_accuracy_score(y_res, pred), roc_auc_score(y_res, proba)

rows = []
for k in [0, 1, 2, 3]:
    Xc = X_all if k == 0 else X_all - X_all @ V[:, :k] @ V[:, :k].T
    site_oob, res_bal, res_auc = evaluate(Xc)
    rows.append({"k": k, "site_oob": site_oob, "res_balacc": res_bal, "res_auc": res_auc})
    print(f"k={k}: site OOB={site_oob:.4f}, res BalAcc={res_bal:.4f}, res AUC={res_auc:.4f}")

df = pd.DataFrame(rows)
df.to_csv(OUT_DIR / "site_deconfound_diagnostic.csv", index=False)
print("\n" + df.to_string(index=False))

# plot: site OOB vs resistance BalAcc across k
fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(df["k"], df["site_oob"], marker='o', color='tab:red', label='site OOB (want lower)')
ax1.set_xlabel("LDA directions removed (k)")
ax1.set_ylabel("site OOB", color='tab:red')
ax1.set_xticks([0, 1, 2, 3])
ax1.grid(True, ls='--', alpha=0.5)
ax2 = ax1.twinx()
ax2.plot(df["k"], df["res_balacc"], marker='s', color='tab:blue', label='res BalAcc (want flat)')
ax2.set_ylabel("resistance BalAcc", color='tab:blue')
ax1.legend(loc='upper left'); ax2.legend(loc='upper right')
plt.title(f"{DRUG_NAME} — LDA site deconfounding tradeoff")
plt.tight_layout(); plt.savefig(OUT_DIR / "site_deconfound_tradeoff.pdf", bbox_inches="tight"); plt.show()


---
**Done.** See `site_deconfound_diagnostic.csv` and `site_deconfound_tradeoff.pdf` in `results/`.

Interpretation:
- `k=0` = baseline (site OOB ~1.0).
- As `k` grows, `site_oob` should fall if the site signal is linear.
- `res_balacc` / `res_auc` should stay near the `k=0` values if site and resistance are separable.
- If `site_oob` barely moves → non-linear site signal. If `res_balacc` tanks → site/resistance entangled.
